# 3.5 图像分类数据集

经典的 MNIST 手写数字数据集作为分类基准过于简单，本节使用更具挑战性的 Fashion-MNIST 服装分类数据集，完整讲解数据集的下载、格式转换、样本可视化与高效批量加载方法，为后续图像分类算法提供数据支撑。

## 3.5.1 读取数据集
通过 torchvision 内置接口下载并加载 Fashion-MNIST 数据集：
- transforms.ToTensor() 将 PIL 格式的灰度图像转换为 32 位浮点张量，同时将像素值从 0~255 归一化到 0~1 区间；
- 数据集分为训练集与测试集，测试集仅用于评估模型性能，不参与训练过程。


In [ ]:
%matplotlib inline
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
from d2l import torch as d2l

d2l.use_svg_display()
# 通过ToTensor实例将图像数据从PIL类型变换成32位浮点数格式，
# 并除以255使得所有像素的数值均在0～1之间
trans = transforms.ToTensor()
mnist_train = torchvision.datasets.FashionMNIST(
    root="C:/Users/Chen Yiliang/PycharmProjects/pythonProject/summer training/data", train=True, transform=trans, download=False)
mnist_test = torchvision.datasets.FashionMNIST(
    root="C:/Users/Chen Yiliang/PycharmProjects/pythonProject/summer training/data", train=False, transform=trans, download=False)

Fashion-MNIST 共包含 10 个服装类别：训练集每类 6000 张图像，总计 60000 张；测试集每类 1000 张图像，总计 10000 张。单张输入图像为 1 通道灰度图，尺寸为 28×28 像素，形状记为 (1, 28, 28)。

In [ ]:
# 查看训练集、测试集样本总数
len(mnist_train), len(mnist_test)

In [ ]:
# 查看单张图像的张量形状
mnist_train[0][0].shape

定义标签映射函数，将 0~9 的数字标签转换为对应的服装类别文本名称，便于可视化与结果解读。10 个类别依次为：T恤、裤子、套衫、连衣裙、外套、凉鞋、衬衫、运动鞋、包、短靴。

In [ ]:
def get_fashion_mnist_labels(labels):  #@save
    """返回Fashion-MNIST数据集的文本标签"""
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
                   'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]

定义图像批量可视化函数，以网格形式绘制多张图像，支持 PyTorch 张量与 PIL 图像两种输入格式，可配置行列数、标题与显示比例。

In [ ]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):  #@save
    """绘制图像列表"""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = d2l.plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        if torch.is_tensor(img):
            # 图片张量
            ax.imshow(img.numpy())
        else:
            # PIL图片
            ax.imshow(img)
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes

从训练集中取出 18 个样本，重塑为 28×28 尺寸后进行可视化，并标注对应文本标签，直观查看数据集内容。

In [ ]:
X, y = next(iter(data.DataLoader(mnist_train, batch_size=18)))
show_images(X.reshape(18, 28, 28), 2, 9, titles=get_fashion_mnist_labels(y));

## 3.5.2 读取小批量
使用 PyTorch 原生 DataLoader 构建高效批量数据迭代器，替代手写迭代器：
- shuffle=True：每个迭代周期自动打乱训练集样本顺序，消除数据排列带来的训练偏差
- num_workers：开启多进程并行读取数据，缓解 IO 瓶颈，避免数据加载阻塞 GPU 训练


In [ ]:
batch_size = 256

def get_dataloader_workers():  #@save
    """使用4个进程来读取数据"""
    return 4

train_iter = data.DataLoader(mnist_train, batch_size, shuffle=True,
                             num_workers=get_dataloader_workers())

使用计时器统计遍历完整训练集的总耗时，验证数据加载性能，确保其不会成为训练流程的瓶颈。

In [ ]:
timer = d2l.Timer()
for X, y in train_iter:
    continue
f'{timer.stop():.2f} sec'

## 3.5.3 整合所有组件
封装 load_data_fashion_mnist 通用函数，一站式完成数据集下载、预处理与迭代器构建，支持通过 resize 参数自定义图像尺寸，最终返回训练集与测试集两个数据迭代器，方便后续章节直接复用。
函数处理逻辑：
1.	构建数据变换流水线：默认包含 ToTensor 归一化，若指定 resize 则在前方插入尺寸调整变换
2.	分别加载训练集与测试集
3.	构造对应 DataLoader：训练集打乱顺序，测试集保持顺序不变以保证评估一致性


In [ ]:
def load_data_fashion_mnist(batch_size, resize=None):  #@save
    """下载Fashion-MNIST数据集，然后将其加载到内存中"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="C:/Users/Chen Yiliang/PycharmProjects/pythonProject/summer training/data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="C:/Users/Chen Yiliang/PycharmProjects/pythonProject/summer training/data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True,
                            num_workers=get_dataloader_workers()),
            data.DataLoader(mnist_test, batch_size, shuffle=False,
                            num_workers=get_dataloader_workers()))

测试尺寸调整功能：指定 resize=64 将图像放大到 64×64，取出一个批量打印张量形状与数据类型，验证函数功能正常。

In [ ]:
train_iter, test_iter = load_data_fashion_mnist(32, resize=64)
for X, y in train_iter:
    print(X.shape, X.dtype, y.shape, y.dtype)
    break